<a href="https://colab.research.google.com/github/yukirochi/Credit-Card-Fraud-Detection/blob/main/Credit_Card_Fraud_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the specific file in the dataset you want to load
file_path = "fraudTest.csv"

# Use dataset_load instead of load_dataset
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "tusharbhadouria/credit-card-fraud-detection",
  file_path,
)

print("First 5 records:\n", df.head())

Using Colab cache for faster access to the 'credit-card-fraud-detection' dataset.
First 5 records:
    Unnamed: 0 trans_date_trans_time  ...  merch_long is_fraud
0           0   2020-06-21 12:14:25  ...  -81.200714        0
1           1   2020-06-21 12:14:33  ... -109.960431        0
2           2   2020-06-21 12:14:53  ...  -74.196111        0
3           3   2020-06-21 12:15:15  ...  -80.883061        0
4           4   2020-06-21 12:15:17  ...  -85.884734        0

[5 rows x 23 columns]


In [57]:


df = df.dropna().drop(columns=['first','last','gender','dob','street','trans_date_trans_time','job','cc_num','trans_num'])
print(f'REMAINING COLUMN \n{df.columns}')

print(f' CHECK FOR NULL VALUES \n{df.isna().sum()}')

print(f' COUNT OF EACH UNIQUE MERCHANT {df['merchant'].value_counts()}')

print(len(df))

REMAINING COLUMN 
Index(['Unnamed: 0', 'merchant', 'category', 'amt', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'unix_time', 'merch_lat', 'merch_long',
       'is_fraud'],
      dtype='object')
 CHECK FOR NULL VALUES 
Unnamed: 0    0
merchant      0
category      0
amt           0
city          0
state         0
zip           0
lat           0
long          0
city_pop      0
unix_time     0
merch_lat     0
merch_long    0
is_fraud      0
dtype: int64
 COUNT OF EACH UNIQUE MERCHANT merchant
fraud_Kilback LLC                        1859
fraud_Cormier LLC                        1597
fraud_Schumm PLC                         1561
fraud_Kuhn LLC                           1521
fraud_Dickinson Ltd                      1519
                                         ... 
fraud_Treutel-King                        323
fraud_Satterfield-Lowe                    319
fraud_Kessler Group                       318
fraud_Jerde-Hermann                       312
fraud_Ritchie, Bradtke and St

In [58]:
from sklearn.preprocessing import LabelEncoder


label = LabelEncoder()

df['merchant'] = label.fit_transform(df['merchant'])
df['state'] = label.fit_transform(df['state'])
df['category'] = label.fit_transform(df['category'])
df['city'] = label.fit_transform(df['city'])



print(df.iloc[1,:])
print(df.head())
print(df.corr()['is_fraud'].sort_values(ascending=False))

Unnamed: 0    1.000000e+00
merchant      5.910000e+02
category      1.000000e+01
amt           2.984000e+01
city          1.600000e+01
state         4.300000e+01
zip           8.400200e+04
lat           4.032070e+01
long         -1.104360e+02
city_pop      3.020000e+02
unix_time     1.371817e+09
merch_lat     3.945050e+01
merch_long   -1.099604e+02
is_fraud      0.000000e+00
Name: 1, dtype: float64
   Unnamed: 0  merchant  category  ...  merch_lat  merch_long  is_fraud
0           0       319        10  ...  33.986391  -81.200714         0
1           1       591        10  ...  39.450498 -109.960431         0
2           2       611         5  ...  40.495810  -74.196111         0
3           3       222         9  ...  28.812398  -80.883061         0
4           4       292        13  ...  44.959148  -85.884734         0

[5 rows x 14 columns]
is_fraud      1.000000
amt           0.182267
category      0.016980
lat           0.005863
merch_lat     0.005812
city          0.001777
merch

In [59]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler



x = df.drop(columns=['is_fraud'])
y = df['is_fraud']

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state= 42)
print(x.columns)


scalar = StandardScaler()
scalar_val_cols = ['amt', 'city_pop', 'lat', 'long']
x_train[scalar_val_cols] = scalar.fit_transform(x_train[scalar_val_cols])

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(x_train, y_train)

y_pred =  rf_model.predict(x_test)

print(classification_report(y_test, y_pred))



Index(['Unnamed: 0', 'merchant', 'category', 'amt', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'unix_time', 'merch_lat', 'merch_long'],
      dtype='object')
              precision    recall  f1-score   support

           0       1.00      0.99      0.99    110718
           1       0.00      0.00      0.00       426

    accuracy                           0.99    111144
   macro avg       0.50      0.50      0.50    111144
weighted avg       0.99      0.99      0.99    111144

